# Device Code login → an **HDF5 dataset**

Sign in with the **Device Authorization Grant** (RFC 8628), then use Lakekeeper's `dataset`
generic-table format (a container for arbitrary files at a vended location) to store an
**HDF5** file. We write the `.h5` locally, upload it to the table's vended S3 location, and
load it back as h5py datasets.

> **Kernel:** use the **Python (pylakekeeper)** kernel (`python/.venv`). Requirements:
> ```sh
> cd python && pip install -e '.[examples]'   # includes h5py + boto3
> ```
> Needs a Keycloak-fronted Lakekeeper and an existing warehouse + namespace.

In [ ]:
import os

KEYCLOAK = os.environ.get("KEYCLOAK", "http://localhost:30080")
REALM = os.environ.get("KEYCLOAK_REALM", "iceberg")
ISSUER = f"{KEYCLOAK}/realms/{REALM}"
DEVICE_AUTHORIZATION_URL = f"{ISSUER}/protocol/openid-connect/auth/device"
TOKEN_URL = f"{ISSUER}/protocol/openid-connect/token"
CLIENT_ID = os.environ.get("OAUTH_CLIENT_ID", "lakekeeper")  # public client
SCOPE = os.environ.get("OAUTH_SCOPE", "openid offline_access")

LAKEKEEPER = os.environ.get("LAKEKEEPER", "http://localhost:8181")
WAREHOUSE_ID = os.environ.get("WAREHOUSE_ID", "")  # warehouse UUID (URL prefix, not name)
PROJECT_ID = os.environ.get("PROJECT_ID") or None
NAMESPACE = os.environ.get("NAMESPACE", "ai.test")
TABLE = os.environ.get("TABLE", "sensor_hdf5")

## Sign in (device code)

In [ ]:
from pylakekeeper import DeviceCodeFlow

try:
    from IPython.display import HTML, display

    def notebook_prompt(p):
        target = p.verification_uri_complete or p.verification_uri
        html = (
            '<div style="padding:12px;border:1px solid #888;border-radius:8px;'
            'font-family:sans-serif;max-width:520px"><b>🔐 Sign in to continue</b><br>'
            f'Open <a href="{target}" target="_blank">{target}</a>'
        )
        if not p.verification_uri_complete and p.user_code:
            html += f' and enter code <code style="font-size:1.2em">{p.user_code}</code>'
        display(HTML(html + "</div>"))
except ImportError:
    notebook_prompt = None

auth = DeviceCodeFlow(
    device_authorization_url=DEVICE_AUTHORIZATION_URL,
    token_url=TOKEN_URL,
    client_id=CLIENT_ID,
    scope=SCOPE,
    prompt=notebook_prompt,
)
print("Signed in ✅", auth.auth_header()[:24], "...")

## Create an HDF5 file
Two datasets (`embeddings`, `id`) plus file-level attributes.

In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np

EMBED_DIM, ROWS = 8, 100
rng = np.random.default_rng(42)
embeddings = rng.standard_normal((ROWS, EMBED_DIM)).astype("float32")
ids = np.arange(ROWS, dtype="int64")

local_h5 = str(Path(tempfile.gettempdir()) / f"{TABLE}.h5")
with h5py.File(local_h5, "w") as f:
    f.create_dataset("embeddings", data=embeddings, compression="gzip")
    f.create_dataset("id", data=ids)
    f.attrs["embedding_dim"] = EMBED_DIM
    f.attrs["rows"] = ROWS
print("wrote", local_h5, "(", os.path.getsize(local_h5), "bytes )")

## Create the `dataset` table and upload the HDF5 file

`format=dataset` registers a container for arbitrary files; `load(vended=True)` returns the
S3 base location + short-lived credentials, which we hand to `boto3` for the upload.

In [ ]:
from urllib.parse import urlparse

import boto3

from pylakekeeper import Client, ConflictError, GenericTableFormat


def s3_client(opts):
    # `lance_storage_options` already uses aws_* keys; endpoint is set for MinIO/SeaweedFS.
    return boto3.client(
        "s3",
        aws_access_key_id=opts["aws_access_key_id"],
        aws_secret_access_key=opts["aws_secret_access_key"],
        aws_session_token=opts.get("aws_session_token"),
        region_name=opts.get("aws_region"),
        endpoint_url=opts.get("aws_endpoint"),
    )


def object_ref(resp):
    parsed = urlparse(resp.location)  # s3://<bucket>/<prefix>
    return parsed.netloc, f"{parsed.path.strip('/')}/{TABLE}.h5"


with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    try:
        c.generic_tables.create(
            NAMESPACE, TABLE, format=GenericTableFormat.DATASET, doc="HDF5 sensor data"
        )
    except ConflictError:
        print(f"{NAMESPACE}.{TABLE} already exists — continuing")
    resp = c.generic_tables.load(NAMESPACE, TABLE, vended=True)

bucket, key = object_ref(resp)
s3_client(resp.lance_storage_options).put_object(
    Bucket=bucket,
    Key=key,
    Body=Path(local_h5).read_bytes(),
    ContentType="application/x-hdf5",
)
print(f"uploaded -> s3://{bucket}/{key}")

## Load it back as a dataset

Re-load for **fresh** credentials, download the object, and open the bytes with `h5py` —
then read the datasets straight into NumPy arrays.

In [ ]:
import io

with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    resp = c.generic_tables.load(NAMESPACE, TABLE, vended=True)  # fresh short-lived creds

bucket, key = object_ref(resp)
raw = s3_client(resp.lance_storage_options).get_object(Bucket=bucket, Key=key)["Body"].read()

with h5py.File(io.BytesIO(raw), "r") as f:
    print("datasets  :", list(f.keys()))
    print("attrs     :", dict(f.attrs))
    emb = f["embeddings"][:]
    ids = f["id"][:]
print("embeddings :", emb.shape, emb.dtype)
print("first ids  :", ids[:5].tolist())